<a href="https://colab.research.google.com/github/AkemjotSingh/csot-ml-astronomy-akemjotsingh/blob/main/week2_mlp_starter_akemjotsingh.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CSoT'26 - ML in Astronomy - Week 2 . Part 2: Your First Neural Network (Starter)

**Goal:** Define a 2-layer fully-connected network (MLP) with `nn.Module`, forward-pass a real batch, set up a loss + optimiser, and run a single optimisation step to watch the loss drop.

**Before you begin:**
1. Switch this notebook to a **GPU runtime** (`Runtime -> Change runtime type -> GPU`).
2. Read [`04-building-models-with-nn-module.md`](../04-building-models-with-nn-module.md) and [`05-loss-functions-and-optimisers.md`](../05-loss-functions-and-optimisers.md).

Replace each `TODO` with working code. **Do not** open the solution until you've genuinely attempted every TODO. (We *set up* training here; the full multi-epoch training loop is Week 3.)

## Step 0 — Re-create the Week 1 data pipeline

Week 2 builds directly on the `DataLoader`s from Week 1. The cells below reproduce that pipeline (download is commented out — uncomment it the first time, exactly as in [`week1_data_solution.ipynb`](../../Week-1/notebooks/week1_data_solution.ipynb)). If you saved `galaxy_data/` to Google Drive in Week 1, just re-mount Drive and point `DATA_ROOT` at it instead of re-downloading.

After this section you should have `train_loader`, `val_loader`, `test_loader`, `train_ds`, and `num_classes`.

In [1]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", device)

Using: cuda


In [2]:
# TODO: paste your Week 1 data pipeline here so that the following names are defined:
#   train_ds, val_ds, test_ds, train_loader, val_loader, test_loader, num_classes
#
# The quickest path is to copy the data-prep cells from
# ../../Week-1/notebooks/week1_data_solution.ipynb (Steps 1-8), then add:
#   num_classes = len(train_ds.classes)
from google.colab import files

files.upload()

os.makedirs("/root/.config/kaggle", exist_ok=True)
os.rename("kaggle.json", "/root/.config/kaggle/kaggle.json")
os.chmod("/root/.config/kaggle/kaggle.json", 0o600)

!pip install kaggle -q
!kaggle datasets download -d jaimetrickz/galaxy-zoo-2-images
!unzip -q galaxy-zoo-2-images.zip -d galaxy_raw

!wget -q "https://gz2hart.s3.amazonaws.com/gz2_hart16.csv.gz" -O galaxy_raw/gz2_hart16.csv.gz
!gunzip galaxy_raw/gz2_hart16.csv.gz

RAW_ROOT   = Path("galaxy_raw")
IMAGES_DIR = RAW_ROOT / "images_gz2" / "images"
DATA_ROOT  = Path("galaxy_data")

Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/jaimetrickz/galaxy-zoo-2-images
License(s): Attribution 4.0 International (CC BY 4.0)
100% 3.06G/3.06G [02:27<00:00, 22.3MB/s]



In [28]:
def high_level_label(gz2_class):
    """Collapse detailed GZ2 codes (Sc2t, Ei, SBb2m, ...) to a few training buckets."""
    if not isinstance(gz2_class, str) or gz2_class == "A":
        return None
    if gz2_class.startswith("E"):
        return "elliptical"
    if gz2_class.startswith("SB"):
        return "spiral_barred"
    if gz2_class.startswith("S"):
        return "spiral"
    return None


def load_labeled_table(mapping_csv, labels_csv):
    mapping = pd.read_csv(mapping_csv)
    labels = pd.read_csv(labels_csv)
    if "dr7objid" in labels.columns:
        labels = labels.rename(columns={"dr7objid": "objid"})
    df = mapping.merge(labels[["objid", "gz2_class"]], on="objid", how="inner")
    df["label"] = df["gz2_class"].map(high_level_label)
    return df.dropna(subset=["label"]).reset_index(drop=True)


def _link_image(src, dst):
    if dst.exists():
        return False
    dst.parent.mkdir(parents=True, exist_ok=True)
    try:
        os.symlink(Path(src).resolve(), dst)
    except OSError:
        import shutil
        shutil.copy2(src, dst)
    return True


def build_split_imagefolder_layout(images_dir, df, out_root, per_class=200,
                                   train_frac=0.70, val_frac=0.15, test_frac=0.15, seed=42):
    images_dir, out_root = Path(images_dir), Path(out_root)
    for label in sorted(df["label"].unique()):
        rows = df[df["label"] == label].sample(frac=1, random_state=seed)
        if len(rows) > per_class:
            rows = rows.head(per_class)
        n = len(rows)
        n_train, n_val = int(train_frac * n), int(val_frac * n)
        splits = {
            "train": rows.iloc[:n_train],
            "val": rows.iloc[n_train:n_train + n_val],
            "test": rows.iloc[n_train + n_val:],
        }
        for split_name, split_rows in splits.items():
            for _, row in split_rows.iterrows():
                src = images_dir / f"{int(row.asset_id)}.jpg"
                dst = out_root / split_name / label / f"{int(row.asset_id)}.jpg"
                if src.exists():
                    _link_image(src, dst)


PER_CLASS = 200  # bump up (e.g. 2000) once everything works

df = load_labeled_table(RAW_ROOT / "gz2_filename_mapping.csv", RAW_ROOT / "gz2_hart16.csv")
build_split_imagefolder_layout(IMAGES_DIR, df, DATA_ROOT, per_class=PER_CLASS)
print(df["label"].value_counts())

label
elliptical       97670
spiral           95849
spiral_barred    45581
Name: count, dtype: int64


In [29]:
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

train_ds = ImageFolder(root=DATA_ROOT / "train", transform=transform)
val_ds   = ImageFolder(root=DATA_ROOT / "val",   transform=transform)
test_ds  = ImageFolder(root=DATA_ROOT / "test",  transform=transform)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

num_classes = len(train_ds.classes)
print("classes      :", train_ds.classes)
print("class_to_idx :", train_ds.class_to_idx)
print("num_classes  :", num_classes)

classes      : ['elliptical', 'spiral', 'spiral_barred']
class_to_idx : {'elliptical': 0, 'spiral': 1, 'spiral_barred': 2}
num_classes  : 3


## Step 1 - Define the model

A 2-layer MLP: flatten -> Linear(12288, 128) -> ReLU -> Linear(128, num_classes). The output layer returns **raw logits** (no softmax - `CrossEntropyLoss` adds it). Don't forget `super().__init__()`.

In [30]:
# TODO: define GalaxyMLP(nn.Module) with:
#   self.flatten = nn.Flatten()
#   self.fc1 = nn.Linear(3*64*64, 128); self.relu = nn.ReLU()
#   self.fc2 = nn.Linear(128, num_classes)
# and a forward(self, x) that runs flatten -> fc1 -> relu -> fc2 and returns logits.
class GalaxyMLP(nn.Module):
    def __init__(self, in_features=3 * 64 * 64, hidden=128, num_classes=3):
        super().__init__()                        # MUST be first
        self.flatten = nn.Flatten()               # (B,3,64,64) -> (B,12288)
        self.fc1 = nn.Linear(in_features, hidden)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden, num_classes) # outputs raw logits

    def forward(self, x):                          # x: (B, 3, 64, 64)
        x = self.flatten(x)                        # (B, 12288)
        x = self.relu(self.fc1(x))                 # (B, 128)
        return self.fc2(x)                         # (B, num_classes)

## Step 2 - Instantiate and move to the device

Use the real `num_classes` from your data, and `.to(device)` so the model lives where the batches will.

In [31]:
# TODO: model = GalaxyMLP(num_classes=num_classes).to(device)
model = GalaxyMLP(num_classes=num_classes).to(device)
print("Model is on:", next(model.parameters()).device)

Model is on: cuda:0


## Step 3 - Inspect the architecture

`print(model)` shows the layers; counting `.parameters()` shows how many weights there are. Notice that the first `Linear` (12288 x 128) dominates - a direct cost of flattening.

In [32]:
# TODO: print(model), then print total and trainable parameter counts.
print(model)
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters     : {total:,}")
print(f"Trainable parameters : {trainable:,}")

GalaxyMLP(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (fc1): Linear(in_features=12288, out_features=128, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=128, out_features=3, bias=True)
)

Total parameters     : 1,573,379
Trainable parameters : 1,573,379


## Step 4 - Forward-pass one real batch

Pull a batch from `train_loader`, move it to the device, and run it through the model. The output should be logits of shape `(batch_size, num_classes)`.

In [33]:
# TODO: images, labels = next(iter(train_loader)); move both to device.
#       logits = model(images); print logits.shape and confirm it's (B, num_classes).
images, labels = next(iter(train_loader))
images, labels = images.to(device), labels.to(device)

logits = model(images)
print("input  :", images.shape)
print("logits :", logits.shape)   # (32, num_classes)
assert logits.shape == (images.shape[0], num_classes)

input  : torch.Size([32, 3, 64, 64])
logits : torch.Size([32, 3])


## Step 5 - Loss and optimiser

`CrossEntropyLoss` consumes raw logits + integer labels. `Adam` with `lr=1e-3` is the sensible default.

In [34]:
# TODO: criterion = nn.CrossEntropyLoss()
#       optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
print(criterion)
print(optimizer)

CrossEntropyLoss()
Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)


## Step 6 - Sanity-check the starting loss

For an untrained model on `C` balanced classes the loss should sit near `ln(C)`. If it's wildly off (or `nan`), suspect label dtype, a stray softmax, or unnormalised inputs.

In [35]:
# TODO: loss = criterion(logits, labels); print loss.item() and compare to math.log(num_classes).
import math

loss = criterion(logits, labels)
print(f"loss            : {loss.item():.4f}")
print(f"ln(num_classes) : {math.log(num_classes):.4f}   <- expected ballpark")

loss            : 1.1215
ln(num_classes) : 1.0986   <- expected ballpark


## Step 7 - One optimisation step (learning, in miniature)

The three lines at the heart of every training loop: clear gradients, backprop, update. Re-evaluate the loss on the *same* batch afterwards - it should drop. (Week 3 repeats this over all batches for many epochs.)

In [36]:
# TODO: model.train()
#       optimizer.zero_grad(); loss.backward(); optimizer.step()
#       recompute the loss on the SAME batch and confirm it decreased.
model.train()
loss_before = criterion(model(images), labels)

optimizer.zero_grad()   # 1. clear old gradients
loss_before.backward()  # 2. backpropagation
optimizer.step()        # 3. update weights

loss_after = criterion(model(images), labels)
print(f"loss before step : {loss_before.item():.4f}")
print(f"loss after  step : {loss_after.item():.4f}")
print("decreased!" if loss_after < loss_before else "did not decrease - check lr / setup")

loss before step : 1.1215
loss after  step : 10.9334
did not decrease - check lr / setup


## Step 8 (stretch) - How big does the model get?

Re-build the MLP with different hidden widths and print the parameter count each time. The first `Linear` (in_features x hidden) dominates - flattening is expensive. A CNN (Week 3) does more with far fewer parameters by sharing weights across the image.

In [37]:
# TODO (optional): rebuild the MLP with hidden widths 64/128/256/512 and print param counts.
for hidden in (64, 128, 256, 512):
    m = GalaxyMLP(hidden=hidden, num_classes=num_classes)
    n = sum(p.numel() for p in m.parameters())
    print(f"hidden={hidden:4d}  ->  {n:,} parameters")

hidden=  64  ->  786,691 parameters
hidden= 128  ->  1,573,379 parameters
hidden= 256  ->  3,146,755 parameters
hidden= 512  ->  6,293,507 parameters


## Reflection *(write 2-3 sentences each)*

1. Your untrained loss should have been near `ln(num_classes)`. Why is that the expected starting value?
2. After one step the loss dropped on that batch. Why is that *not yet* the same as 'the model is trained'?
3. Compare the MLP's parameter count to what you'd expect a CNN to need. Why does flattening make the first layer so large?

Answer 1:An untrained model spreads probability roughly equally over `C` classes, so the true class gets ~`1/C` probability and the cross-entropy is `-ln(1/C) = ln(C)`.

Answer 2:We only lowered the loss on a *single* batch. Real training repeats `zero_grad -> backward -> step` over *all* batches for many epochs, and is judged on *held-out* data - that's Week 3.

Answer 3:Flattening connects every one of 12 288 inputs to every hidden unit, so `fc1` alone holds `12288 x hidden` weights. A CNN reuses small filters across the whole image, so it captures spatial structure with far fewer parameters.